# 933점 이후 — 하이퍼파라미터 튜닝 + fold 10 + seed 앙상블

## 현재까지 경과

| 버전 | 리더보드 | 핵심 |
|---|---|---|
| 818점 | 818.84 | `TimeSeriesSplit` fold 5 단독 |
| 897점 | 897.07 | `StratifiedKFold` 5-fold, 트랙맨 `exact`(test에서 0) |
| **933점** | **932.96** | 트랙맨 `asof` 모드 (2025가 2024 값을 받음) — **현재 기준선** |
| (실패) | 930.00 | 타자 측 트랙맨 피처 추가 — 롤백됨 |

933점 구성(트랙맨 `asof`, CatBoost 단독, 5-fold)은 그대로 두고, 이번엔
**모델 자체를 더 잘 뽑아내는 방향**으로 개선한다.

## 이 노트북에서 하는 것

1. **Optuna 하이퍼파라미터 탐색** — `depth=6, lr=0.05`는 한 번도 튜닝 안 한 기본값이었다.
   빠른 3-fold 서브샘플 채점으로 40회 탐색한다.
2. **fold 5 → 10** — 각 fold가 90%를 학습하게 되어, 개별 모델 품질이 오르고
   평균 대상도 늘어 분산이 더 줄어든다.
3. **seed 앙상블 (3개)** — 같은 설정을 다른 무작위 분할로 3번 반복해서 평균낸다.
   총 10 × 3 = **30개 모델**을 학습해서 평균한다.

셋 다 **933점 대비 구조를 안 바꾸고 안전하게 쌓는 개선**이다 (새로운 피처를
추가하는 게 아니라, 같은 정보를 더 안정적으로 뽑아내는 것).

## ⚠️ 검증에 대한 경고 (이전과 동일)

`StratifiedKFold`를 쓰므로 로컬 홀드아웃 검증은 성능 지표로 쓸 수 없다.
다만 **Optuna 튜닝(Cell 6a)만은 예외**다 — "이 하이퍼파라미터가 주어진 데이터를
얼마나 잘 맞히는가"는 `StratifiedKFold`로도 정직하게 잴 수 있으므로, 이 부분은
로컬 판단이 유효하다. Cell 6b 이후(fold 10 × seed 3 최종 학습)의 OOF와
933점 대비 실제 개선 여부는 **리더보드로만** 판단한다.


## [Cell 0] 라이브러리 및 설정

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ---------------- 설정 ----------------
# 'asof'  : merge_asof backward. 2025 test 행이 가장 최근(2024) 트랙맨 값을 받는다.
#           학습/추론이 동일한 규칙을 쓰므로 원칙적으로 더 타당하다.
# 'exact' : (season, month) 정확 일치 + fillna(0). 900점 버전과 완전히 동일한 동작
#           (트랙맨에 2025가 없어 test에서는 전부 0이 된다).
TRACKMAN_MODE = 'asof'

N_SPLITS = 10                 # 5 -> 10 (각 fold가 90%를 학습, 평균 대상도 늘어 분산 감소)
SEEDS = [42, 202, 2024]       # seed 앙상블 (3개) -> 총 N_SPLITS * len(SEEDS) = 30개 모델
N_OPTUNA_TRIALS = 40          # 하이퍼파라미터 탐색 횟수
print(f"TRACKMAN_MODE = {TRACKMAN_MODE} | N_SPLITS = {N_SPLITS} | SEEDS = {SEEDS}")


## [Cell 1] Step 1~14 — 학습/추론 공용 소스

`STEPS_SRC` 문자열 하나를 노트북과 `script.py`가 공유한다.
전처리가 두 갈래로 갈라지는 것 자체를 불가능하게 만드는 장치
(이전에 이 불일치로 0점이 난 적이 있다).


In [ ]:
STEPS_SRC = r"""
def step1_basic_features(df):
    df_proc = df.copy()
    df_proc['is_weekend_day_game'] = np.where(
        (df_proc['game_month'].isin([4, 5, 9, 10])) & (df_proc['game_dayofweek'].isin([5, 6])), 1.0, 0.0)
    df_proc['is_heat_wave_game'] = np.where(df_proc['game_month'].isin([7, 8]), 1.0, 0.0)
    return df_proc


def step2_pitcher_role_features(df):
    df_proc = df.copy()
    df_proc['is_pure_starter'] = np.where(df_proc['inning'] == 1, 1.0, 0.0)
    df_proc['is_long_relief'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] >= (df_proc['inning'] - 1) * 12), 1.0, 0.0)
    df_proc['is_short_relief'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] < (df_proc['inning'] - 1) * 12), 1.0, 0.0)
    return df_proc


def step3_matchup_features(df):
    df_proc = df.copy()
    if 'pitcher_hand' in df_proc.columns and 'batter_hand' in df_proc.columns:
        df_proc['is_same_hand'] = np.where(df_proc['pitcher_hand'] == df_proc['batter_hand'], 1.0, 0.0)
    return df_proc


def step4_refined_count_features(df):
    df_proc = df.copy()
    b, s = df_proc['balls_before'], df_proc['strikes_before']
    df_proc['is_first_pitch'] = np.where((b == 0) & (s == 0), 1.0, 0.0)
    df_proc['is_full_count'] = np.where((b == 3) & (s == 2), 1.0, 0.0)
    pitcher_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
    batter_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
    neutral = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
    df_proc['count_advantage'] = np.select(
        [pitcher_ahead, batter_ahead, neutral], ['Pitcher', 'Batter', 'Neutral'], default='None')
    df_proc['is_waste_pitch_sit'] = np.where(((b == 0) & (s == 2)) | ((b == 1) & (s == 2)), 1.0, 0.0)
    df_proc['is_must_strike_sit'] = np.where(((b == 3) & (s == 0)) | ((b == 3) & (s == 1)), 1.0, 0.0)
    return df_proc


def step5_pitches_per_inning(df):
    df_proc = df.copy()
    df_proc['pitches_per_inning'] = df_proc['asof_pitcher_n'] / df_proc['inning'].clip(lower=1)
    return df_proc


def step6_combined_runner_features(df):
    df_proc = df.copy()
    df_proc['is_risp'] = df_proc['base_state'].astype(str).apply(
        lambda x: 1.0 if ('2' in x) or ('3' in x) else 0.0)
    df_proc['is_strict_inherited_runner'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] < 5) & (df_proc['num_runners_on'] > 0), 1.0, 0.0)
    df_proc['is_self_risp'] = np.where(
        (df_proc['asof_pitcher_n'] >= 15) & (df_proc['is_risp'] == 1.0), 1.0, 0.0)
    li_filled = df_proc['li'].fillna(0)
    df_proc['risp_pressure_index'] = df_proc['is_risp'] * li_filled
    df_proc['is_steal_threat_sit'] = np.where(
        (df_proc['runner_on_1b'] == 1) & (df_proc['runner_on_2b'] == 0)
        & (df_proc['score_diff_pitcher_team'].abs() <= 3), 1.0, 0.0)
    return df_proc


def step7_bayesian_smoothing(df, prior_mean=0.64):
    df_proc = df.copy()
    C = 50
    if 'asof_pitcher_success_rate' in df_proc.columns and 'asof_pitcher_n' in df_proc.columns:
        n = df_proc['asof_pitcher_n']
        curr = df_proc['asof_pitcher_success_rate']
        df_proc['smoothed_pitcher_success_rate'] = (n * curr + C * prior_mean) / (n + C)
    return df_proc


def step8_batter_toughness_features(df):
    df_proc = df.copy()
    if 'asof_batter_success_rate' in df_proc.columns and 'asof_batter_middle_rate' in df_proc.columns:
        df_proc['tough_batter_index'] = (1.0 - df_proc['asof_batter_success_rate']) * (1.0 - df_proc['asof_batter_middle_rate'])
    return df_proc


def step9_garbage_time_features(df):
    df_proc = df.copy()
    df_proc['is_garbage_time'] = np.where(df_proc['score_diff_pitcher_team'].abs() >= 7, 1.0, 0.0)
    df_proc['garbage_time_index'] = df_proc['score_diff_pitcher_team'].abs() / (10 - df_proc['inning']).clip(lower=1)
    return df_proc


def step10_recent_form_momentum(df):
    df_proc = df.copy()
    tc = ['asof_pitcher_prev1_game_success_rate',
          'asof_pitcher_prev3_game_success_rate',
          'asof_pitcher_prev5_game_success_rate']
    if all(c in df_proc.columns for c in tc):
        p1, p3, p5 = df_proc[tc[0]], df_proc[tc[1]], df_proc[tc[2]]
        df_proc['momentum_short'] = p1 - p3
        df_proc['momentum_mid'] = p1 - p5
        df_proc['is_heating_up'] = np.where((p1 > p3) & (p3 > p5), 1.0, 0.0)
        df_proc['is_cooling_down'] = np.where((p1 < p3) & (p3 < p5), 1.0, 0.0)
    return df_proc


def step11_veteran_and_pressure_features(df):
    df_proc = df.copy()
    df_proc['is_rookie'] = np.where(df_proc['asof_pitcher_n'] < 684, 1.0, 0.0)
    df_proc['is_veteran'] = np.where(df_proc['asof_pitcher_n'] > 3725, 1.0, 0.0)
    li_filled = df_proc['li'].fillna(0)
    df_proc['rookie_crisis_risk'] = df_proc['is_rookie'] * li_filled
    df_proc['veteran_clutch_ability'] = df_proc['is_veteran'] * li_filled
    return df_proc


def step12_first_pitch_tendency(df):
    df_proc = df.copy()
    if 'asof_pitcher_fastball_rate' in df_proc.columns and 'asof_pitcher_strike_rate' in df_proc.columns:
        if 'is_first_pitch' in df_proc.columns:
            df_proc['first_pitch_fastball_strike_idx'] = (
                df_proc['is_first_pitch'] * df_proc['asof_pitcher_fastball_rate'] * df_proc['asof_pitcher_strike_rate'])
    return df_proc


def step13_sac_fly_threat(df):
    df_proc = df.copy()
    is_3b = df_proc['base_state'].astype(str).apply(lambda x: 1.0 if '3' in x else 0.0)
    df_proc['is_sac_fly_threat'] = np.where(
        (is_3b == 1.0) & (df_proc['outs_before'] < 2)
        & (df_proc['score_diff_pitcher_team'].abs() <= 3), 1.0, 0.0)
    return df_proc


def step14_convert_to_category(df):
    df_proc = df.copy()
    original_cat_cols = ['pitcher_id', 'batter_id', 'pitcher_team_id', 'batter_team_id',
                         'pitcher_hand', 'batter_hand', 'base_state', 'stadium',
                         'pitch_name', 'top_bottom', 'game_type']
    created_cat_cols = ['is_weekend_day_game', 'is_heat_wave_game', 'is_pure_starter',
                        'is_long_relief', 'is_short_relief', 'is_same_hand', 'is_first_pitch',
                        'is_full_count', 'count_advantage', 'is_waste_pitch_sit',
                        'is_must_strike_sit', 'is_risp', 'is_strict_inherited_runner',
                        'is_self_risp', 'is_steal_threat_sit', 'is_sac_fly_threat',
                        'is_garbage_time', 'is_rookie', 'is_veteran',
                        'is_heating_up', 'is_cooling_down']
    all_cat_cols = [c for c in original_cat_cols + created_cat_cols if c in df_proc.columns]
    for c in all_cat_cols:
        df_proc[c] = df_proc[c].astype('category')
    return df_proc
"""

exec(STEPS_SRC)
print("step1~14 정의 완료")


## [Cell 2] Step 15~18 — Trackman 요약 (학습 시 1회만 실행)

In [ ]:
def step15_prep_trackman_data(trackman_df, pitcher_map_df):
    tm = pd.merge(trackman_df, pitcher_map_df[['pitcher_id', 'pitcher_trackman_id']],
                  on='pitcher_trackman_id', how='inner')
    b, s = tm['balls_before'], tm['strikes_before']
    p_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
    b_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
    neu = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
    tm['count_advantage'] = np.select([p_ahead, b_ahead, neu],
                                       ['Pitcher', 'Batter', 'Neutral'], default='None')
    tm['pitch_group'] = tm['pitch_type_group'].astype(str).str.lower()
    return tm[tm['pitch_group'].isin(['fastball', 'breaking', 'offspeed'])].copy()


def step16_calc_expected_difficulty(tm):
    groups = ['fastball', 'breaking', 'offspeed']
    sit = tm.groupby(['season', 'game_month', 'pitcher_id', 'count_advantage', 'pitch_group']
                     ).size().unstack(fill_value=0).reset_index()
    for c in groups:
        if c not in sit.columns:
            sit[c] = 0
    sit = sit.sort_values(by=['pitcher_id', 'count_advantage', 'season', 'game_month'])
    g = sit.groupby(['pitcher_id', 'count_advantage'])
    sit['past_fb'] = g['fastball'].cumsum() - sit['fastball']
    sit['past_br'] = g['breaking'].cumsum() - sit['breaking']
    sit['past_off'] = g['offspeed'].cumsum() - sit['offspeed']
    tot = sit['past_fb'] + sit['past_br'] + sit['past_off']
    sit['past_total'] = tot
    sit['exp_fb_prob'] = np.where(tot > 0, sit['past_fb'] / tot, 0)
    sit['exp_br_prob'] = np.where(tot > 0, sit['past_br'] / tot, 0)
    sit['exp_off_prob'] = np.where(tot > 0, sit['past_off'] / tot, 0)

    dm = tm.groupby(['season', 'game_month', 'pitcher_id', 'pitch_group'])[['rel_height', 'rel_side']].std()
    dm['diff_score'] = dm['rel_height'] + dm['rel_side']
    dm = dm.reset_index()
    dp = dm.pivot_table(index=['season', 'game_month', 'pitcher_id'],
                        columns='pitch_group', values='diff_score', fill_value=np.nan).reset_index()
    for c in groups:
        if c not in dp.columns:
            dp[c] = 0
    dp = dp.sort_values(by=['pitcher_id', 'season', 'game_month'])
    gd = dp.groupby(['pitcher_id'])
    dp['past_fb_diff'] = gd['fastball'].transform(lambda x: x.shift(1).expanding().mean())
    dp['past_br_diff'] = gd['breaking'].transform(lambda x: x.shift(1).expanding().mean())
    dp['past_off_diff'] = gd['offspeed'].transform(lambda x: x.shift(1).expanding().mean())

    res = pd.merge(sit, dp, on=['season', 'game_month', 'pitcher_id'], how='left')
    res['expected_control_difficulty'] = (res['exp_fb_prob'] * res['past_fb_diff']
                                          + res['exp_br_prob'] * res['past_br_diff']
                                          + res['exp_off_prob'] * res['past_off_diff'])
    return res[['season', 'game_month', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']]


def step17_calc_pitch_speed(tm):
    fb = tm[tm['pitch_group'] == 'fastball']
    sp = fb.groupby(['season', 'game_month', 'pitcher_id'])['rel_speed'].mean().reset_index()
    sp = sp.sort_values(by=['pitcher_id', 'season', 'game_month'])
    sp['past_fb_speed_mean'] = sp.groupby(['pitcher_id'])['rel_speed'].transform(
        lambda x: x.shift(1).expanding().mean())
    return sp[['season', 'game_month', 'pitcher_id', 'past_fb_speed_mean']]


def step18_calc_pitch_consistency_by_group(tm):
    groups = ['fastball', 'breaking', 'offspeed']
    metrics = ['rel_height_std', 'rel_side_std', 'extension_std',
               'spin_rate_std', 'vert_break_std', 'horz_break_std']
    cm = tm.groupby(['season', 'game_month', 'pitcher_id', 'pitch_group']).agg(
        rel_height_std=('rel_height', 'std'), rel_side_std=('rel_side', 'std'),
        extension_std=('extension', 'std'), spin_rate_std=('spin_rate', 'std'),
        vert_break_std=('induced_vert_break', 'std'), horz_break_std=('horz_break', 'std')
    ).reset_index()
    pv = cm.pivot_table(index=['season', 'game_month', 'pitcher_id'],
                        columns='pitch_group', values=metrics, fill_value=np.nan)
    pv.columns = [f"{grp}_{val}" for val, grp in pv.columns]
    pv = pv.reset_index()
    for pg in groups:
        for m in metrics:
            if f"{pg}_{m}" not in pv.columns:
                pv[f"{pg}_{m}"] = np.nan
    pv = pv.sort_values(by=['pitcher_id', 'season', 'game_month'])
    g = pv.groupby(['pitcher_id'])
    out_cols = ['season', 'game_month', 'pitcher_id']
    for pg in groups:
        for m in metrics:
            src, dst = f"{pg}_{m}", f"past_{pg}_{m}"
            pv[dst] = g[src].transform(lambda x: x.shift(1).expanding().mean())
            out_cols.append(dst)
    return pv[out_cols]


## [Cell 3] 통합 파이프라인

In [ ]:
def run_full_pipeline(train_df, trackman_df, pitcher_map, trackman_mode='asof'):
    print(f"파이프라인 시작 (trackman_mode={trackman_mode})...")
    df_proc = train_df.copy()

    df_proc = step1_basic_features(df_proc)
    df_proc = step2_pitcher_role_features(df_proc)
    df_proc = step3_matchup_features(df_proc)
    df_proc = step4_refined_count_features(df_proc)
    df_proc = step5_pitches_per_inning(df_proc)
    df_proc = step6_combined_runner_features(df_proc)

    prior_mean = float(df_proc['asof_pitcher_success_rate'].mean())
    print(f"  prior_mean = {prior_mean:.6f}")

    df_proc = step7_bayesian_smoothing(df_proc, prior_mean=prior_mean)
    df_proc = step8_batter_toughness_features(df_proc)
    df_proc = step9_garbage_time_features(df_proc)
    df_proc = step10_recent_form_momentum(df_proc)
    df_proc = step11_veteran_and_pressure_features(df_proc)
    df_proc = step12_first_pitch_tendency(df_proc)
    df_proc = step13_sac_fly_threat(df_proc)

    if 'count_advantage' not in df_proc.columns:
        b, s = df_proc['balls_before'], df_proc['strikes_before']
        p_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
        b_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
        neu = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
        df_proc['count_advantage'] = np.select([p_ahead, b_ahead, neu],
                                                ['Pitcher', 'Batter', 'Neutral'], default='None')

    tm_base = step15_prep_trackman_data(trackman_df, pitcher_map)
    feat_diff = step16_calc_expected_difficulty(tm_base)
    feat_speed = step17_calc_pitch_speed(tm_base)
    feat_rp = step18_calc_pitch_consistency_by_group(tm_base)
    rp_value_cols = [c for c in feat_rp.columns if c.startswith('past_')]

    if trackman_mode == 'asof':
        for f in [feat_diff, feat_speed, feat_rp]:
            f['time_idx'] = f['season'] * 100 + f['game_month']
            f.sort_values('time_idx', inplace=True)
        df_proc['time_idx'] = df_proc['season'] * 100 + df_proc['game_month']
        df_proc = df_proc.sort_values('time_idx')

        df_proc = pd.merge_asof(
            df_proc,
            feat_diff[['time_idx', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']],
            on='time_idx', by=['pitcher_id', 'count_advantage'], direction='backward')
        df_proc = pd.merge_asof(
            df_proc, feat_speed[['time_idx', 'pitcher_id', 'past_fb_speed_mean']],
            on='time_idx', by='pitcher_id', direction='backward')
        df_proc = pd.merge_asof(
            df_proc, feat_rp[['time_idx', 'pitcher_id'] + rp_value_cols],
            on='time_idx', by='pitcher_id', direction='backward')
        df_proc = df_proc.drop(columns=['time_idx'])
    else:
        df_proc = pd.merge(df_proc, feat_diff,
                           on=['season', 'game_month', 'pitcher_id', 'count_advantage'], how='left')
        df_proc = pd.merge(df_proc, feat_speed,
                           on=['season', 'game_month', 'pitcher_id'], how='left')
        df_proc = pd.merge(df_proc, feat_rp,
                           on=['season', 'game_month', 'pitcher_id'], how='left')
        for c in ['expected_control_difficulty', 'past_fb_speed_mean'] + rp_value_cols:
            if c in df_proc.columns:
                df_proc[c] = df_proc[c].fillna(0)

    df_proc = step14_convert_to_category(df_proc)
    print("파이프라인 완료.")
    return df_proc.reset_index(drop=True), prior_mean, feat_diff, feat_speed, feat_rp


## [Cell 4] 데이터 로드 및 실행

In [ ]:
DATA_DIR = "/kaggle/input/datasets/homekeggle/aimers/open/data"
MAP_PATH = "/kaggle/input/datasets/homekeggle/aimers/pitcher_id_mapping.csv"

df_train = pd.read_csv(f"{DATA_DIR}/train.csv")
df_trackman = pd.read_csv(f"{DATA_DIR}/trackman_history.csv")
pitcher_id_mapping = pd.read_csv(MAP_PATH)
print("train:", df_train.shape, "| trackman:", df_trackman.shape)


In [ ]:
df_processed, PRIOR_MEAN, feat_diff, feat_speed, feat_rp = run_full_pipeline(
    df_train, df_trackman, pitcher_id_mapping, trackman_mode=TRACKMAN_MODE)
print("df_processed:", df_processed.shape)


## [Cell 5] 추론용 아티팩트 저장

In [ ]:
os.makedirs("model", exist_ok=True)

with open("model/train_constants.json", "w") as f:
    json.dump({"prior_mean": PRIOR_MEAN, "trackman_mode": TRACKMAN_MODE}, f)
print(f"train_constants.json  prior_mean={PRIOR_MEAN:.6f}  trackman_mode={TRACKMAN_MODE}")

# 트랙맨 테이블은 step16~18 출력 그대로 저장 (dropna/dedup 금지)
_rp = [c for c in feat_rp.columns if c.startswith('past_')]
_diff_cols = ['season', 'game_month', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']
_speed_cols = ['season', 'game_month', 'pitcher_id', 'past_fb_speed_mean']
_rp_cols = ['season', 'game_month', 'pitcher_id'] + _rp
if TRACKMAN_MODE == 'asof':
    for f_, extra in [(feat_diff, _diff_cols), (feat_speed, _speed_cols), (feat_rp, _rp_cols)]:
        if 'time_idx' not in f_.columns:
            f_['time_idx'] = f_['season'] * 100 + f_['game_month']
    _diff_cols = ['time_idx'] + _diff_cols
    _speed_cols = ['time_idx'] + _speed_cols
    _rp_cols = ['time_idx'] + _rp_cols

feat_diff[_diff_cols].to_csv("model/feat_diff.csv", index=False)
feat_speed[_speed_cols].to_csv("model/feat_speed.csv", index=False)
feat_rp[_rp_cols].to_csv("model/feat_rp.csv", index=False)
print(f"feat_diff {len(feat_diff):,} / feat_speed {len(feat_speed):,} / feat_rp {len(feat_rp):,}")

# 'None' 라운드트립 검증: 0-0/3-2 카운트를 뜻하는 실제 문자열인데
# pd.read_csv 기본 설정은 NaN으로 읽어버려 merge가 전량 실패한다.
_NA = ['', 'NaN', 'nan', 'NULL', 'null', 'NA', 'N/A', 'n/a']
_chk = pd.read_csv("model/feat_diff.csv", keep_default_na=False, na_values=_NA)
_n_none = (_chk['count_advantage'].astype(str) == 'None').sum()
_bad = pd.read_csv("model/feat_diff.csv")['count_advantage'].isna().sum()
print(f"\n'None' 행 {_n_none:,}개 — 기본 read_csv로는 {_bad:,}개가 NaN이 됨 (script.py는 na_values 명시)")
assert _n_none > 0, "'None' 값이 사라졌습니다"


## [Cell 6a] Optuna 하이퍼파라미터 탐색 (안전, 로컬 판단 가능)

**이건 리더보드 도박이 아니다.** `StratifiedKFold` 자체는 "미래 시즌 예측력"을 정직하게
재지 못하지만, "이 하이퍼파라미터가 주어진 학습 데이터를 얼마나 잘 맞히는가"(OOF Brier)는
`StratifiedKFold`로도 정직하게 잴 수 있다. 그래서 튜닝은 로컬에서 안전하게 결정한다.

**속도를 위한 설계**: 매 trial마다 10-fold를 다 돌리면 40 trial × 10 fold라 너무 느리다.
대신 데이터의 30%만 표본으로 뽑아 3-fold로 빠르게 채점한다. 최종 학습(Cell 6b)은 이렇게
찾은 파라미터로 전체 데이터 × 10-fold × 3-seed를 학습한다.


In [ ]:
try:
    import optuna
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)
    import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import brier_score_loss
from catboost import CatBoostClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

target_col = 'control_success'
drop_cols = [target_col, 'row_id', 'pitcher_id', 'batter_id', 'time_idx']
feature_cols = [c for c in df_processed.columns if c not in drop_cols]

X_full = df_processed[feature_cols].copy()
y_full = df_processed[target_col].copy()
for col in [c for c in X_full.columns if X_full[c].dtype.name in ['category', 'object']]:
    X_full[col] = X_full[col].astype(str).astype('category')
cat_features = [c for c in X_full.columns if X_full[c].dtype.name == 'category']

with open("model/selected_features.json", "w") as f:
    json.dump(list(feature_cols), f)
print(f"피처 {len(feature_cols)}개 (범주형 {len(cat_features)}개)")

# 탐색용 30% 서브샘플 (계층 유지)
# skf.split()은 (train_idx, test_idx) 순서로 반환한다. 30%에 가까운 건 test_idx(약 33%) 쪽이므로
# 두 번째 원소를 받는다 (첫 번째를 받으면 train_idx=약 67%가 되어 의도보다 훨씬 커진다).
_, _sub_idx = next(StratifiedKFold(n_splits=3, shuffle=True, random_state=0).split(X_full, y_full))
X_sub, y_sub = X_full.iloc[_sub_idx], y_full.iloc[_sub_idx]
print(f"Optuna 탐색용 서브샘플: {len(X_sub):,}행 (전체의 약 {len(X_sub)/len(X_full):.0%})")


def objective(trial):
    params = {
        "iterations": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 0.5, 3.0),
        "eval_metric": "Logloss",
        "cat_features": cat_features,
        "random_seed": 42,
        "task_type": "GPU",
        "early_stopping_rounds": 50,
    }
    skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)
    briers = []
    for tr_idx, val_idx in skf3.split(X_sub, y_sub):
        model = CatBoostClassifier(**params)
        model.fit(X_sub.iloc[tr_idx], y_sub.iloc[tr_idx],
                  eval_set=(X_sub.iloc[val_idx], y_sub.iloc[val_idx]), verbose=0)
        p = model.predict_proba(X_sub.iloc[val_idx])[:, 1]
        briers.append(brier_score_loss(y_sub.iloc[val_idx], p))
    return float(np.mean(briers))


print("\n=== Optuna 탐색 시작 ===")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

BEST_PARAMS = dict(study.best_params)
BEST_PARAMS["iterations"] = 1000
BEST_PARAMS["eval_metric"] = "Logloss"
BEST_PARAMS["task_type"] = "GPU"
BEST_PARAMS["early_stopping_rounds"] = 50
BEST_PARAMS["cat_features"] = cat_features  # 누락돼 있었음 -- 없으면 Cell 6b의 .fit()에서 CatBoostError로 크래시함

print(f"\n최적 Brier: {study.best_value:.5f}")
print("최적 파라미터:", BEST_PARAMS)

with open("model/best_params.json", "w") as f:
    json.dump(BEST_PARAMS, f, indent=2)

## [Cell 6b] 최종 학습 — fold 10 × seed 3 = 30개 모델

Optuna로 찾은 파라미터로 전체 데이터에 대해 학습한다. `StratifiedKFold`를 seed별로
독립적으로 3번 돌려서 (10-fold × 3-seed = 30개 모델), 최종 제출은 이 30개를 평균한다.
fold 수를 늘리고 seed를 다양화하는 건 각각 **분산을 줄이는 효과**라 안전하게 쌓인다.

⚠️ 모델 30개를 학습하므로 기존(5개)보다 시간이 훨씬 오래 걸린다 (GPU 기준 체감 3~6배).


In [ ]:
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import brier_score_loss
from sklearn.calibration import CalibratedClassifierCV
from catboost import CatBoostClassifier

X, y = X_full, y_full  # Cell 6a에서 만든 전체 데이터 재사용


def extract_isotonic(cv_obj):
    cc = cv_obj.calibrated_classifiers_[0]
    if hasattr(cc, 'calibrators'):
        return cc.calibrators[0]
    if hasattr(cc, 'calibrators_'):
        return cc.calibrators_[0]
    raise AttributeError("보정기를 찾을 수 없습니다.")


def brier_and_skill(y_t, p, tag=""):
    b = brier_score_loss(y_t, p)
    r = np.mean(y_t)
    naive = r * (1 - r)
    skill = 1 - b / naive
    print(f"  {tag:<20} Brier={b:.5f}  Skill={skill:+.3%}  (리더보드 환산 ≈ {skill*100000:,.0f})")
    return skill


seed_oof_raw = {s: np.zeros(len(X)) for s in SEEDS}
seed_oof_cal = {s: np.zeros(len(X)) for s in SEEDS}

print(f"\n=== 최종 학습: {N_SPLITS}-fold x {len(SEEDS)}-seed = {N_SPLITS * len(SEEDS)}개 모델 ===")
for seed in SEEDS:
    print(f"\n########## SEED {seed} ##########")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"[ seed {seed} / fold {fold+1}/{N_SPLITS} ]", end=" ")
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        params = dict(BEST_PARAMS)
        params["random_seed"] = seed
        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)
        raw = model.predict_proba(X_val)[:, 1]
        seed_oof_raw[seed][val_idx] = raw
        model.save_model(f"model/cb_fold_{seed}_{fold+1}.cbm")

        _c = CalibratedClassifierCV(model, method='isotonic', cv='prefit')
        _c.fit(X_val, y_val)
        iso = extract_isotonic(_c)
        cal = iso.predict(raw)
        seed_oof_cal[seed][val_idx] = cal
        joblib.dump(iso, f"model/isotonic_fold_{seed}_{fold+1}.pkl")

        print(f"Brier(cal)={brier_score_loss(y_val, cal):.5f}")

print(f"\n학습 및 저장 완료 ({N_SPLITS * len(SEEDS)}개 모델)")

print("\n=== seed별 전체 OOF ===")
for seed in SEEDS:
    brier_and_skill(y.to_numpy(), seed_oof_cal[seed], f"seed {seed}")

print("\n=== seed 평균 OOF (이게 최종 제출과 가장 비슷한 조합) ===")
avg_oof_cal = np.mean([seed_oof_cal[s] for s in SEEDS], axis=0)
brier_and_skill(y.to_numpy(), avg_oof_cal, "seed 평균")
print("\n주의: 이 OOF도 StratifiedKFold 기반이라 절대 성능 지표가 아니다.")
print("      933점(fold5/seed1 구성) 대비 개선 여부는 리더보드로만 판단할 것.")


## [Cell 7] `script.py` 생성 및 zip 패키징

In [ ]:
SCRIPT_TEMPLATE = r"""import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import json
import traceback
import numpy as np
import pandas as pd
import joblib
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# ================= 학습과 문자 단위로 동일한 전처리 =================
__STEPS__
# ====================================================================


def main():
    data_dir = None
    for path in ["data", "open", "./data", "./open", "open/data"]:
        if os.path.exists(os.path.join(path, "test.csv")):
            data_dir = path
            break
    if data_dir is None:
        raise FileNotFoundError("평가용 데이터를 찾을 수 없습니다.")

    df_test = pd.read_csv(os.path.join(data_dir, "test.csv"))
    row_ids = df_test['row_id'].copy() if 'row_id' in df_test.columns else df_test.index

    constants_path = os.path.join("model", "train_constants.json")
    if not os.path.exists(constants_path):
        raise RuntimeError("model/train_constants.json이 없습니다. prior_mean을 알 수 없어 중단합니다.")
    with open(constants_path, "r") as f:
        prior_mean = float(json.load(f)["prior_mean"])

    df_proc = step1_basic_features(df_test)
    df_proc = step2_pitcher_role_features(df_proc)
    df_proc = step3_matchup_features(df_proc)
    df_proc = step4_refined_count_features(df_proc)
    df_proc = step5_pitches_per_inning(df_proc)
    df_proc = step6_combined_runner_features(df_proc)
    df_proc = step7_bayesian_smoothing(df_proc, prior_mean=prior_mean)
    df_proc = step8_batter_toughness_features(df_proc)
    df_proc = step9_garbage_time_features(df_proc)
    df_proc = step10_recent_form_momentum(df_proc)
    df_proc = step11_veteran_and_pressure_features(df_proc)
    df_proc = step12_first_pitch_tendency(df_proc)
    df_proc = step13_sac_fly_threat(df_proc)

    if 'count_advantage' not in df_proc.columns:
        b, s = df_proc['balls_before'], df_proc['strikes_before']
        p_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
        b_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
        neu = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
        df_proc['count_advantage'] = np.select([p_ahead, b_ahead, neu],
                                                ['Pitcher', 'Batter', 'Neutral'], default='None')

    # ---------- 트랙맨 병합 ----------
    # 학습 때 쓴 방식(train_constants.json의 trackman_mode)을 그대로 따라간다.
    #   asof  : merge_asof backward. 2025 test 행은 가장 최근(2024) 값을 받는다.
    #   exact : (season, month) 정확 일치 merge + fillna(0).
    #           트랙맨에 2025가 없으므로 test에서는 전부 0이 된다 (900점 버전의 동작).
    with open(constants_path, "r") as f:
        _const = json.load(f)
    trackman_mode = _const.get("trackman_mode", "asof")

    _NA = ['', 'NaN', 'nan', 'NULL', 'null', 'NA', 'N/A', 'n/a']
    fd_path = os.path.join("model", "feat_diff.csv")
    fs_path = os.path.join("model", "feat_speed.csv")
    fr_path = os.path.join("model", "feat_rp.csv")
    has_tm = all(os.path.exists(p) for p in [fd_path, fs_path, fr_path])

    if has_tm:
        # 'None'은 0-0/3-2 카운트를 뜻하는 실제 문자열인데 pandas 기본 설정은
        # 이를 NaN으로 읽어버린다. 그러면 by= 매칭이 전량 실패한다.
        feat_diff = pd.read_csv(fd_path, keep_default_na=False, na_values=_NA)
        feat_speed = pd.read_csv(fs_path, keep_default_na=False, na_values=_NA)
        feat_rp = pd.read_csv(fr_path, keep_default_na=False, na_values=_NA)
        feat_diff['count_advantage'] = feat_diff['count_advantage'].astype(str)
        rp_value_cols = [c for c in feat_rp.columns if c.startswith('past_')]

        if trackman_mode == "asof":
            df_proc['time_idx'] = df_proc['season'] * 100 + df_proc['game_month']
            df_proc['__orig'] = np.arange(len(df_proc))
            df_proc = df_proc.sort_values('time_idx')
            feat_diff = feat_diff.sort_values('time_idx')
            feat_speed = feat_speed.sort_values('time_idx')
            feat_rp = feat_rp.sort_values('time_idx')

            df_proc = pd.merge_asof(
                df_proc,
                feat_diff[['time_idx', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']],
                on='time_idx', by=['pitcher_id', 'count_advantage'], direction='backward')
            df_proc = pd.merge_asof(
                df_proc, feat_speed[['time_idx', 'pitcher_id', 'past_fb_speed_mean']],
                on='time_idx', by='pitcher_id', direction='backward')
            df_proc = pd.merge_asof(
                df_proc, feat_rp[['time_idx', 'pitcher_id'] + rp_value_cols],
                on='time_idx', by='pitcher_id', direction='backward')

            df_proc = df_proc.sort_values('__orig').drop(columns=['__orig', 'time_idx'])
        else:
            df_proc = pd.merge(
                df_proc,
                feat_diff[['season', 'game_month', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']],
                on=['season', 'game_month', 'pitcher_id', 'count_advantage'], how='left')
            df_proc = pd.merge(
                df_proc, feat_speed[['season', 'game_month', 'pitcher_id', 'past_fb_speed_mean']],
                on=['season', 'game_month', 'pitcher_id'], how='left')
            df_proc = pd.merge(
                df_proc, feat_rp[['season', 'game_month', 'pitcher_id'] + rp_value_cols],
                on=['season', 'game_month', 'pitcher_id'], how='left')
            for c in ['expected_control_difficulty', 'past_fb_speed_mean'] + rp_value_cols:
                if c in df_proc.columns:
                    df_proc[c] = df_proc[c].fillna(0)

    df_proc = step14_convert_to_category(df_proc)

    with open("model/selected_features.json", "r") as f:
        selected_features = json.load(f)
    for col in selected_features:
        if col not in df_proc.columns:
            df_proc[col] = np.nan
    df_features = df_proc[selected_features].copy()

    # CatBoost는 cat_features에 실제 NaN을 허용하지 않는다 (학습과 동일 처리)
    for col in df_features.columns:
        if df_features[col].dtype.name in ['category', 'object']:
            df_features[col] = df_features[col].astype(str).astype('category')

    # ---------- 추론: 모델 전체 평균 ----------
    # StratifiedKFold(shuffle=True) x 여러 seed로 학습했으므로 모든 모델이
    # 대등한 실력을 가진다 -> 균등 평균이 순수한 분산 감소로 이어진다.
    # 파일명에서 seed/fold 조합을 실제로 스캔한다 (개수를 하드코딩하지 않음 ->
    # N_SPLITS/SEEDS를 나중에 바꿔도 script.py 수정이 필요 없다).
    import glob
    preds = []
    cb_paths = sorted(glob.glob(os.path.join("model", "cb_fold_*.cbm")))
    for cb_path in cb_paths:
        stem = os.path.splitext(os.path.basename(cb_path))[0]  # cb_fold_{seed}_{fold}
        suffix = stem[len("cb_fold_"):]  # {seed}_{fold}
        model = CatBoostClassifier()
        model.load_model(cb_path)
        names = list(model.feature_names_)
        df_in = df_features.copy()
        for col in names:
            if col not in df_in.columns:
                df_in[col] = np.nan
        raw = model.predict_proba(df_in[names])[:, 1]
        iso_path = os.path.join("model", "isotonic_fold_%s.pkl" % suffix)
        if os.path.exists(iso_path):
            raw = joblib.load(iso_path).predict(raw)
        preds.append(raw)

    if len(preds) == 0:
        raise RuntimeError(
            "모델을 하나도 로드하지 못했습니다. cwd=%s, model=%s"
            % (os.getcwd(), sorted(os.listdir('model')) if os.path.isdir('model') else '(없음)'))

    final_preds = np.mean(preds, axis=0)
    if np.isnan(final_preds).any():
        final_preds = np.nan_to_num(final_preds, nan=prior_mean)
    final_preds = np.clip(final_preds, 0.01, 0.99)

    os.makedirs("output", exist_ok=True)
    submission = pd.DataFrame({"row_id": row_ids, "control_success": final_preds})

    sample_path = os.path.join(data_dir, "sample_submission.csv")
    if os.path.exists(sample_path):
        sample = pd.read_csv(sample_path)
        sample['row_id'] = sample['row_id'].astype(str)
        submission['row_id'] = submission['row_id'].astype(str)
        sample = sample.drop(columns=['control_success'], errors='ignore')
        sample = sample.merge(submission, on='row_id', how='left')
        sample['control_success'] = sample['control_success'].fillna(prior_mean)
        sample.to_csv("output/submission.csv", index=False)
    else:
        submission.to_csv("output/submission.csv", index=False)


if __name__ == "__main__":
    try:
        main()
    except Exception:
        os.makedirs("output", exist_ok=True)
        with open("output/error_log.txt", "w", encoding="utf-8") as f:
            f.write(traceback.format_exc())
        raise
"""

script_content = SCRIPT_TEMPLATE.replace("__STEPS__", STEPS_SRC)

with open("script.py", "w", encoding="utf-8") as f:
    f.write(script_content)
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("catboost\n")

import ast
ast.parse(script_content)
print(f"script.py 생성 완료 ({len(script_content):,}자, 문법 검사 통과)")


In [ ]:
import zipfile
import glob

cb_files = sorted(glob.glob("model/cb_fold_*.cbm"))
iso_files = sorted(glob.glob("model/isotonic_fold_*.pkl"))
expected_n = N_SPLITS * len(SEEDS)
print(f"모델 파일 {len(cb_files)}개 발견 (기대 {expected_n}개)")
if len(cb_files) != expected_n or len(iso_files) != expected_n:
    raise RuntimeError(
        f"모델 파일 개수가 예상과 다릅니다 (cb={len(cb_files)}, iso={len(iso_files)}, "
        f"기대={expected_n}). Cell 6b가 끝까지 정상 실행됐는지 확인하세요."
    )

REQUIRED = (
    ["script.py", "requirements.txt"]
    + [os.path.relpath(p) for p in cb_files]
    + [os.path.relpath(p) for p in iso_files]
    + ["model/selected_features.json", "model/train_constants.json", "model/best_params.json",
       "model/feat_diff.csv", "model/feat_speed.csv", "model/feat_rp.csv"]
)

missing = [p for p in REQUIRED if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"다음 파일이 없습니다: {missing}")

ZIP_PATH = "submit_tuned.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in REQUIRED:
        zf.write(p, arcname=p)
print(f"{ZIP_PATH} 생성 완료 — {len(REQUIRED)}개 파일")


## [Cell 8] 파이프라인 버그 탐지 (성능 측정용 아님)

`script.py`를 실제로 실행해 `submission.csv`가 정상적으로 나오는지 확인한다.

**⚠️ 여기 나오는 Skill 수치는 성능 지표로 쓰면 안 된다.** `StratifiedKFold`로 학습해
모델이 이 홀드아웃 행들을 이미 학습에 사용했으므로, 점수가 크게 부풀려진다.

이 셀의 목적은 오직 다음 세 가지 확인이다.
1. 스크립트가 크래시 없이 끝까지 도는가
2. `submission.csv`의 행 수와 `row_id`가 맞는가
3. 예측 분포가 상식적인가 (`std`가 0에 가깝거나 전부 0.01이면 버그)


In [ ]:
import shutil, subprocess, sys

SANDBOX = "validation_sandbox"
_n = min(50000, len(df_train))
sample = df_train.sample(_n, random_state=1).reset_index(drop=True)

if os.path.exists(SANDBOX):
    shutil.rmtree(SANDBOX)
os.makedirs(f"{SANDBOX}/data", exist_ok=True)
sample.to_csv(f"{SANDBOX}/data/test.csv", index=False)
shutil.copy2("script.py", f"{SANDBOX}/script.py")
shutil.copytree("model", f"{SANDBOX}/model")

print(f"{_n:,}행으로 script.py 실행 중...")
res = subprocess.run([sys.executable, "script.py"], cwd=SANDBOX, capture_output=True, text=True)
print("종료 코드:", res.returncode)
if res.stderr.strip():
    print("--- stderr ---")
    print(res.stderr[-3000:])

sub_path = f"{SANDBOX}/output/submission.csv"
if not os.path.exists(sub_path):
    err = f"{SANDBOX}/output/error_log.txt"
    if os.path.exists(err):
        print(open(err, encoding="utf-8").read())
    raise RuntimeError("submission.csv가 생성되지 않았습니다.")

sub = pd.read_csv(sub_path)
p = sub["control_success"].to_numpy()
print(f"\n행 수: {len(sub):,} (기대 {_n:,})  결측: {np.isnan(p).sum()}")
print(f"예측 분포: min={p.min():.4f} max={p.max():.4f} mean={p.mean():.4f} std={p.std():.4f}")

ok = (len(sub) == _n) and (np.isnan(p).sum() == 0) and (p.std() > 0.005)
print("\n[통과] 파이프라인 정상." if ok else "\n[실패] 위 수치를 확인하세요.")
print("(반복: 이 셀은 버그 탐지용이며 성능 판단용이 아닙니다.)")


---

## 실행 및 제출 전략

1. Cell 0~7 순서대로 실행 (Cell 6a: 튜닝, Cell 6b: 최종 30개 모델 학습 — 시간이 꽤 걸림)
2. `submit_tuned.zip` 제출해서 933점 대비 개선 여부 확인
3. 개선됐으면 이 구성을 새 기준선으로 삼고, 다음 단계(`asof_*` 피처 재설계)로 진행

## 그 다음 개선 후보

| 순위 | 항목 | 이유 |
|---|---|---|
| 1 | **`asof_*` 피처 직접 재설계** | 카운트/구종/좌우별 세분화, 경기 단위가 아닌 최근 N구 이동평균. 작업량은 크지만 로드맵상 핵심 승부처 |
| 2 | **LightGBM 재투입** | 이번엔 CatBoost와 대등한 조건(StratifiedKFold)에서 블렌드 비율을 OOF로 탐색 |
| 3 | **`middle_rate`/`reverse_rate` 분해 기반 보조모델** | 제구 실패의 3가지 유형(가운데/크게벗어남/반대방향)을 따로 예측해서 스태킹 |

마감(9/2)까지 최소 이틀은 신규 시도 없이 버퍼로 남겨둘 것.
